<img src="https://geo-credito-rural.github.io/_static/logo.jpeg" align="right" width="64" />

# <span style="color:#336699">Impedimentos Sociais, Ambientais e Climáticos</span>

<hr style="border:2px solid #0077b9;">
<div style="text-align: left;">
    <a href="https://nbviewer.jupyter.org/github/brazil-data-cube/code-gallery/blob/master/jupyter/Python/stac/stac-image-processing.ipynb"><img src="https://raw.githubusercontent.com/jupyter/design/master/logos/Badges/nbviewer_badge.svg" align="center"/></a>
</div>

<br/>

<div style="text-align: center;font-size: 90%;">
     Gabriel Sansigolo, Thales Sehn Körting, Gilberto Queiroz, Karine Ferreira, Marcos Adami
    <br/><br/>
    Divisão de Observação da Terra e Geoinformática, Instituto Nacional de Pesquisas Espaciais (INPE)
    <br/>
    Avenida dos Astronautas, 1758, Jardim da Granja, São José dos Campos, SP 12227-010, Brazil
    <br/><br/>
    Contato: <a href="https://geo-credito-rural.github.io/">Equipe - Geo Credito Rural</a>
    <br/><br/>
    Última atualização: 11 de abril de 2026
</div>


<br/>

</div>

Esse notebook aborda de forma simplificada as restrições legais para o acesso ao crédito rural com base em critérios sociais, ambientais e climáticos.

# <span style="color:#336699">Contextualização</span>
<hr style="border:1px solid #0077b9;">

Com as recentes atualizações do Conselho Monetário Nacional (como as Resoluções CMN Nº 5.193/2024, 5.267/2025 e 5.268/2025), a verificação de financiamentos tornou-se mais rigorosa. Hoje, exige-se a identificação da propriedade via Cadastro Ambiental Rural (CAR) e o monitoramento obrigatório por sensoriamento remoto para áreas superiores a 300 hectares, garantindo que os recursos financiados respeitem a conformidade socioambiental.

Na atividade prática de hoje, vamos analisar os impedimentos definidos para esses empreendimentos.

# <span style="color:#336699">1 - Cadastro Ambiental Rural (CAR)</span>
<hr style="border:1px solid #0077b9;">

O Cadastro Ambiental Rural (CAR) tem o objetivo de integrar as informações ambientais das propriedades e posses rurais, compondo base de dados para controle, monitoramento, planejamento ambiental e econômico e combate ao desmatamento. Ele é um registro público eletrônico declaratório, obrigatório para todos os imóveis rurais, e foi criado pela Lei Nº 12.651, de 25 de maio de 2012 [35] no âmbito do Sistema Nacional de Informação sobre Meio Ambiente (SINIMA).

De acordo com a Resolução CMN Nº 5.193/2024:

> 2 - Para os fins de que trata esta Seção, a identificação do imóvel rural onde se situa o empreendimento objeto do
crédito rural será realizada de acordo com as informações registradas no Sistema Nacional de Cadastro
Ambiental Rural (Sicar). (Res CMN 5.193 art 1º)

Uma representação ilustrativa em forma de diagrama da restrição pode ser observada na Figura 1:

![car](https://i.imgur.com/Nz5bfQH.png "Cadastro Ambiental Rural")

Prática: vamos simular a verificação de sobreposição.




**Configuração e Exemplo COM Sobreposição**

Neste cenário, criamos um polígono para o empreendimento que intercepta uma das áreas do CAR.

In [ ]:
from shapely.geometry import Polygon
import matplotlib.pyplot as plt
import requests, zipfile, io
import geopandas as gpd

In [ ]:
empreendimento = Polygon([(0, 0), (1, 0), (1, 1), (0, 1)])

In [ ]:
lista_car = [
    Polygon([(0.5, 0.5), (1.5, 0.5), (1.5, 1.5), (0.5, 1.5)]),
    Polygon([(2, 2), (3, 2), (3, 3), (2, 3)]),
    Polygon([(4, 4), (5, 4), (5, 5), (4, 5)]),
    Polygon([(6, 6), (7, 6), (7, 7), (6, 7)]),
    Polygon([(8, 8), (9, 8), (9, 9), (8, 9)])
]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

x, y = empreendimento.exterior.xy
ax.plot(x, y, color='#1f77b4', linewidth=2, label='Empreendimento')
ax.fill(x, y, color='#1f77b4', alpha=0.3)

for i, poly in enumerate(lista_car):
    x, y = poly.exterior.xy
    label = 'CAR' if i == 0 else None
    ax.plot(x, y, color='#d62728', linewidth=2, label=label)
    ax.fill(x, y, color='#d62728', alpha=0.3)

ax.set_aspect('equal')
ax.set_title('Visualização: Empreendimento vs. Lista CAR')
ax.set_xlabel('Longitude / X')
ax.set_ylabel('Latitude / Y')
ax.legend()
plt.grid(True, linestyle='--', alpha=0.6)

plt.show()

In [ ]:
# Passa por todas as áreas da lista car, pegando a posição (i) e a geometria (car).
for i, car in enumerate(lista_car):

    # Checa se a área do empreendimento cruza ou encosta na área do car atual.
    if empreendimento.intersects(car):

        # Calcula o tamanho da área que as duas áreas se cruzam.
        sobreposicao_area = empreendimento.intersection(car).area

        print(f"O empreendimento sobrepõe a área CAR {i}! Área da sobreposição: {sobreposicao_area:.2f} unidades.")

    # Executa este bloco caso não haja nenhum cruzamento entre as áreas.
    else:
        print(f"Sem sobreposição.")

**Configuração e Exemplo SEM Sobreposição**

Aqui, usamos a mesma lista de 5 áreas do CAR, mas movemos a coordenada do empreendimento para bem longe, garantindo que ele passe limpo por todas as verificações.

In [ ]:
empreendimento = Polygon([(15, 15), (16, 15), (16, 16), (15, 16)])

In [ ]:
lista_car = [
    Polygon([(0.5, 0.5), (1.5, 0.5), (1.5, 1.5), (0.5, 1.5)]),
    Polygon([(2, 2), (3, 2), (3, 3), (2, 3)]),
    Polygon([(4, 4), (5, 4), (5, 5), (4, 5)]),
    Polygon([(6, 6), (7, 6), (7, 7), (6, 7)]),
    Polygon([(8, 8), (9, 8), (9, 9), (8, 9)])
]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

x, y = empreendimento.exterior.xy
ax.plot(x, y, color='#1f77b4', linewidth=2, label='Empreendimento')
ax.fill(x, y, color='#1f77b4', alpha=0.3)

for i, poly in enumerate(lista_car):
    x, y = poly.exterior.xy
    label = 'CAR' if i == 0 else None
    ax.plot(x, y, color='#d62728', linewidth=2, label=label)
    ax.fill(x, y, color='#d62728', alpha=0.3)

ax.set_aspect('equal')
ax.set_title('Visualização: Empreendimento vs. Lista CAR')
ax.set_xlabel('Longitude / X')
ax.set_ylabel('Latitude / Y')
ax.legend()
plt.grid(True, linestyle='--', alpha=0.6)

plt.show()

In [ ]:
# Passa por todas as áreas da lista car, pegando a posição (i) e a geometria (car).
for i, car in enumerate(lista_car):

    # Checa se a área do empreendimento cruza ou encosta na área do car atual.
    if empreendimento.intersects(car):

        # Calcula o tamanho da área que as duas áreas se cruzam.
        sobreposicao_area = empreendimento.intersection(car).area

        print(f"O empreendimento sobrepõe a área CAR {i}! Área da sobreposição: {sobreposicao_area:.2f} unidades.")

    # Executa este bloco caso não haja nenhum cruzamento entre as áreas.
    else:
        print(f"Sem sobreposição.")

**Exemplo com dados reais**

**Configuração e Exemplo COM Sobreposição**

In [ ]:
r = requests.get("https://raw.githubusercontent.com/GSansigolo/shapefiles/refs/heads/main/mini_car_ba.zip")

zipfile.ZipFile(io.BytesIO(r.content)).extractall("./mini_car_ba.zip")
file_path = "./mini_car_ba.zip/mini_car_ba.shp"

mini_car = gpd.read_file(file_path)

In [ ]:
r = requests.get("https://raw.githubusercontent.com/GSansigolo/shapefiles/refs/heads/main/poligono_1.zip")

zipfile.ZipFile(io.BytesIO(r.content)).extractall("./poligono_1.zip")
file_path = "./poligono_1.zip/poligono_1.shp"

poligono_1 = gpd.read_file(file_path)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(8, 8))

mini_car.plot(ax=ax, color='#d62728', alpha=0.3)
mini_car.boundary.plot(ax=ax, color='#d62728', linewidth=2)

poligono_1.plot(ax=ax, color='#1f77b4', alpha=0.3)
poligono_1.boundary.plot(ax=ax, color='#1f77b4', linewidth=2)

ax.set_aspect('equal')
ax.set_title('Visualização: Empreendimento vs. Lista CAR')
ax.set_xlabel('Longitude / X')
ax.set_ylabel('Latitude / Y')
plt.grid(True, linestyle='--', alpha=0.6)

legenda_empreendimento = mpatches.Patch(color='#1f77b4', alpha=0.5, label='Empreendimento')
legenda_car = mpatches.Patch(color='#d62728', alpha=0.5, label='CAR')
ax.legend(handles=[legenda_empreendimento, legenda_car])
plt.show()

In [ ]:
poligono_1 = poligono_1.to_crs(mini_car.crs)
geom_empreendimento = poligono_1.geometry.iloc[0]

mini_car_plano = mini_car.to_crs(epsg=6933)
geom_empreendimento_plana = poligono_1.to_crs(epsg=6933).geometry.iloc[0]

sobreposicoes = mini_car.intersects(geom_empreendimento)
areas = mini_car_plano.intersection(geom_empreendimento_plana).area

for i, sobrepoe in sobreposicoes.items():
    if sobrepoe:
        print(f"O empreendimento sobrepõe a área CAR {i}! Área da sobreposição: {areas[i]:.2f} m².")

**Configuração e Exemplo sem Sobreposição**

In [ ]:
r = requests.get("https://raw.githubusercontent.com/GSansigolo/shapefiles/refs/heads/main/poligono_2.zip")

zipfile.ZipFile(io.BytesIO(r.content)).extractall("./poligono_2.zip")
file_path = "./poligono_2.zip/poligono_2.shp"

poligono_2 = gpd.read_file(file_path)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(8, 8))

mini_car.plot(ax=ax, color='#d62728', alpha=0.3)
mini_car.boundary.plot(ax=ax, color='#d62728', linewidth=2)

poligono_2.plot(ax=ax, color='#1f77b4', alpha=0.3)
poligono_2.boundary.plot(ax=ax, color='#1f77b4', linewidth=2)

ax.set_aspect('equal')
ax.set_title('Visualização: Empreendimento vs. Lista CAR')
ax.set_xlabel('Longitude / X')
ax.set_ylabel('Latitude / Y')
plt.grid(True, linestyle='--', alpha=0.6)

legenda_empreendimento = mpatches.Patch(color='#1f77b4', alpha=0.5, label='Empreendimento')
legenda_car = mpatches.Patch(color='#d62728', alpha=0.5, label='CAR')
ax.legend(handles=[legenda_empreendimento, legenda_car])
plt.show()

In [ ]:
poligono_2 = poligono_2.to_crs(mini_car.crs)
geom_empreendimento = poligono_2.geometry.iloc[0]

mini_car_plano = mini_car.to_crs(epsg=6933)
geom_empreendimento_plana = poligono_2.to_crs(epsg=6933).geometry.iloc[0]

sobreposicoes = mini_car.intersects(geom_empreendimento)
areas = mini_car_plano.intersection(geom_empreendimento_plana).area

area_minima = 1.0

for i, sobrepoe in sobreposicoes.items():
    if sobrepoe and areas[i] > area_minima:
        print(f"O empreendimento sobrepõe a área CAR {i}! Área da sobreposição: {areas[i]:.2f} m².")
